# Übung 20 — Datenbeschaffung, Aufbereitung & EDA Schritt für Schritt zum Prüfungsniveau AP02 (Lösung)

Dieses Notebook führt dich in **9 Stufen** zum Niveau der Prüfung
*AP02 Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse*:

| Stufe | Thema | Prüfungsteil |
|---|---|---|
| 1 | BeautifulSoup-Grundlagen an einem Mini-HTML | C |
| 2 | Produktlisten scrapen, Duplikate erkennen | B |
| 3 | Rohtext mit Regex in Zahlen umwandeln | B |
| 4 | Fehlende Werte: zählen, MCAR / MAR / MNAR | A |
| 5 | EDA: Verteilung, Lagemasse, Ausreisser | A, B |
| 6 | Klassen bilden: `cut`, `qcut`, `crosstab` | C |
| 7 | JSON, `merge`, Korrelation, Streudiagramm | A, C |
| 8 | Tabellen von einer Detailseite extrahieren | C |
| 9 | Konzeptfragen mit Bezug zu den Daten | A |

Danach bist du bereit für die **Probeprüfung** `21_probepruefung_ap02.ipynb`.

Ergänze die Zellen mit `# TODO`. Die Zellen **Selbstkontrolle** prüfen dein Ergebnis.

## Hinweise zu den Daten

Alle Daten stammen vom **fiktiven** Schweizer Online-Shop **«VeloMarkt»** und liegen lokal in `Data/velomarkt/`.
Es ist kein Internetzugriff nötig. Spaltenbeschreibungen: `Data/velomarkt/README.md`.

| Datei | Format | Inhalt |
|---|---|---|
| `html/shop_page_1.html` … `shop_page_3.html` | HTML | gespeicherte Produktlisten-Seiten (Paginierung) |
| `html/product_detail.html` | HTML | Detailseite eines E-Bikes (Preisverlauf, technische Daten, Lieferung) |
| `reviews.json` | JSON | gespeicherte Antwort der Reviews-API: Bewertungen pro `product_id` |
| `velos_clean.csv` | CSV | aufbereitete Shop-Daten — **inhaltlich nicht geprüft** |

## Setup

In [ ]:
# Libraries
import os
import re
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Daten von «VeloMarkt» liegen hier:
DATA = 'Data/velomarkt'
print(os.getcwd())

---
## Stufe 1 — BeautifulSoup-Grundlagen

Das folgende Mini-HTML hat denselben Aufbau wie die Shop-Seiten. Öffne zum Vergleich auch
`Data/velomarkt/html/shop_page_1.html` im Editor.

In [ ]:
mini_html = """
<p class="result-count">2 Artikel</p>
<div class="product-card" data-product-id="VM-1">
  <span class="brand">Cube</span> <span class="model">Hyde</span>
  <li class="weight">14,2\xa0kg</li>
  <span class="price">CHF\xa0899.–</span>
</div>
<div class="product-card" data-product-id="VM-2">
  <span class="brand">Stromer</span> <span class="model">ST5</span>
  <li class="weight">27,9\xa0kg</li>
  <span class="price">CHF\xa011'990.–</span>
</div>
"""
mini = BeautifulSoup(mini_html, 'html.parser')

### 1.1 Elemente finden
1. Wie viele Produktkarten (`div.product-card`) gibt es? → `anzahl_karten`
2. Erstelle eine Liste `ids` mit dem Attribut `data-product-id` jeder Karte.
3. Erstelle eine Liste `marken` mit dem Text aller `span.brand`.

In [ ]:
karten = mini.select('div.product-card')
anzahl_karten = len(karten)
ids = [karte['data-product-id'] for karte in karten]
marken = [span.get_text() for span in mini.select('span.brand')]
print(anzahl_karten, ids, marken)

In [ ]:
# Selbstkontrolle 1.1
assert anzahl_karten == 2 and ids == ['VM-1', 'VM-2'] and marken == ['Cube', 'Stromer']
print('1.1 richtig ✔')

### 1.2 Text sauber auslesen
Lies den Preistext jeder Karte aus. Gib den Text einmal **roh** aus (`repr()` zeigt unsichtbare Zeichen) und
erstelle dann die Liste `preise_text`, in der das geschützte Leerzeichen `\xa0` entfernt und der Text mit
`.strip()` bereinigt ist.

In [ ]:
for span in mini.select('span.price'):
    print(repr(span.get_text()))

preise_text = [span.get_text().replace('\xa0', '').strip() for span in mini.select('span.price')]
preise_text

In [ ]:
# Selbstkontrolle 1.2
assert preise_text == ['CHF899.–', "CHF11'990.–"]
print('1.2 richtig ✔')

### 1.3 `select` vs. `select_one`
`select()` liefert **immer eine Liste** (evtl. leer), `select_one()` das **erste** passende Element oder `None`.
Lies mit `select_one` den Text von `p.result-count` und wandle die Zahl darin in einen `int` um → `artikel_total`.

In [ ]:
text_count = mini.select_one('p.result-count').get_text()
artikel_total = int(re.findall(r'\d+', text_count)[0])
artikel_total

In [ ]:
# Selbstkontrolle 1.3
assert artikel_total == 2
print('1.3 richtig ✔')

---
## Stufe 2 — Produktlisten scrapen und Duplikate erkennen

### 2.1 Wie viele Artikel führt der Shop?
Lies aus `shop_page_1.html` die Angabe in `p.result-count` als Zahl → `artikel_shop`.

In [ ]:
with open(f'{DATA}/html/shop_page_1.html', 'r', encoding='utf-8') as file:
    soup_1 = BeautifulSoup(file.read(), 'html.parser')

artikel_shop = int(re.findall(r'\d+', soup_1.select_one('p.result-count').get_text())[0])
artikel_shop

In [ ]:
# Selbstkontrolle 2.1
assert artikel_shop == 120
print('2.1 richtig ✔')

### 2.2 Alle drei Seiten scrapen → `df_raw`
Der Code ist vorgegeben (gleicher Aufbau wie in der Prüfung). Führe ihn aus und vergleiche die Anzahl Zeilen mit
`artikel_shop`.

In [ ]:
# Initialize list for the extracted rows
rows = []

# Loop over the three listing pages
for page in [1, 2, 3]:

    # Read html from file
    with open(f'{DATA}/html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        html_content = file.read()

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Locate the product cards and extract values
    for card in soup.select('div.product-card'):
        row_data = [card['data-product-id'],
                    card.select('span.brand')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.model')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.category')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.weight')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.battery')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.range')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.price')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.availability')[0].get_text().replace('\xa0', '').strip()]
        rows.append(row_data)

# Create a DataFrame
df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'category', 'weight_raw',
                                     'battery_raw', 'range_raw', 'price_raw', 'availability'])

# Show dimensions and first rows
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

### 2.3 Duplikate finden und entfernen
1. Wie viele `product_id` kommen mehrfach vor? (`duplicated()`) → `anzahl_duplikate`
2. Welche IDs sind betroffen? → Liste `doppelte_ids` (sortiert)
3. Entferne die Duplikate anhand von `product_id` und setze den Index neu (`reset_index(drop=True)`) → `df_unique`

*Frage:* Woher kommen die Duplikate? Schau dir an, auf welchen Seiten die betroffenen IDs stehen.

In [ ]:
anzahl_duplikate = df_raw['product_id'].duplicated().sum()
doppelte_ids = sorted(df_raw.loc[df_raw['product_id'].duplicated(), 'product_id'])
df_unique = df_raw.drop_duplicates(subset='product_id').reset_index(drop=True)
print(anzahl_duplikate, doppelte_ids, df_unique.shape)
# Die letzten 6 Produkte von Seite 2 erscheinen am Anfang von Seite 3 nochmals
# (überlappende Paginierung, z. B. weil sich das Sortiment während des Scrapens verändert hat).

In [ ]:
# Selbstkontrolle 2.3
assert anzahl_duplikate == 6 and len(doppelte_ids) == 6
assert df_unique.shape[0] == artikel_shop and df_unique.index.tolist() == list(range(120))
print('2.3 richtig ✔')

---
## Stufe 3 — Rohtext mit Regex in Zahlen umwandeln

### 3.1 Was liefert `re.findall`?
Sage zuerst voraus, was herauskommt, und prüfe dann:

In [ ]:
print(re.findall(r'\d+', "CHF899.–"))
print(re.findall(r'\d+', "CHF1'249.–"))
print(re.findall(r'\d+', "Preis auf Anfrage"))
print(re.findall(r'\d+,\d+|\d+', "25,4kg"))

**Erklärung:**

`\d+` findet zusammenhängende Ziffernfolgen. Der Apostroph als Tausendertrennzeichen **unterbricht** die Ziffernfolge:
Aus `CHF1'249.–` werden die zwei Treffer `['1', '249']`. Wer nur das erste Element `[0]` nimmt, erhält **1 statt 1249**.
Deshalb vorher `.replace("'", "")`. Bei «Preis auf Anfrage» ist die Liste leer, und `[0]` würde einen `IndexError`
auslösen.

### 3.2 Eine robuste Umwandlungsfunktion
Schreibe die Funktion `preis_zu_zahl(text)`: Sie entfernt den Apostroph, sucht die erste Ziffernfolge und gibt sie als
`int` zurück — oder `None`, wenn keine Zahl vorhanden ist.

In [ ]:
def preis_zu_zahl(text):
    ziffern = re.findall(r'\d+', text.replace("'", ''))
    if len(ziffern) == 0:
        return None
    return int(ziffern[0])

In [ ]:
# Selbstkontrolle 3.2
assert preis_zu_zahl("CHF899.–") == 899
assert preis_zu_zahl("CHF11'990.–") == 11990
assert preis_zu_zahl("Preis auf Anfrage") is None
print('3.2 richtig ✔')

### 3.3 Preise umwandeln — und die Index-Falle
1. Wende `preis_zu_zahl` auf `df_unique['price_raw']` an und speichere das Ergebnis als Spalte `price_chf`
   (Datentyp `Int64`, der auch fehlende Werte erlaubt) → `df_33`.
2. Berechne Anzahl fehlender Preise und den Median.

**Achtung, Index-Falle:** Weist man einer Spalte eine **neue `pd.Series`** zu, richtet pandas die Werte nach dem
**Index** aus, nicht nach der Position. Hat der DataFrame nach `drop_duplicates()` Lücken im Index (z. B. 0–79, 86–125),
landen Werte in falschen Zeilen und es entstehen NaN. Darum `reset_index(drop=True)` (wie in 2.3) oder direkt
`.apply()` auf der Spalte verwenden.

In [ ]:
df_33 = df_unique.copy()
df_33['price_chf'] = df_33['price_raw'].apply(preis_zu_zahl).astype('Int64')

anzahl_ohne_preis = df_33['price_chf'].isnull().sum()
median_preis = df_33['price_chf'].median()
print(anzahl_ohne_preis, median_preis)

In [ ]:
# Selbstkontrolle 3.3
assert anzahl_ohne_preis == 6 and median_preis == 3698.0
print('3.3 richtig ✔')

### 3.4 Gewicht, Akku und Reichweite
Wandle in `df_33` die Rohspalten in Zahlen um:
- `weight_kg` aus `weight_raw` (z. B. `25,4kg` → 25.4, `k. A.` → NaN) — Komma durch Punkt ersetzen!
- `battery_wh` aus `battery_raw` (z. B. `800Wh` → 800, `–` → NaN)
- `range_km` aus `range_raw` (z. B. `bis 145km` → 145, `–` → NaN)

*Tipp:* `.str.extract(r'(\d+,\d+|\d+)', expand=False)` liefert die erste Zahl als Text (oder NaN),
`pd.to_numeric(..., errors='coerce')` wandelt in Zahlen um.

In [ ]:
df_33['weight_kg'] = pd.to_numeric(
    df_33['weight_raw'].str.extract(r'(\d+,\d+|\d+)', expand=False).str.replace(',', '.'),
    errors='coerce')
df_33['battery_wh'] = pd.to_numeric(df_33['battery_raw'].str.extract(r'(\d+)', expand=False), errors='coerce')
df_33['range_km'] = pd.to_numeric(df_33['range_raw'].str.extract(r'(\d+)', expand=False), errors='coerce')

df_33[['weight_kg', 'battery_wh', 'range_km']].describe()

In [ ]:
# Selbstkontrolle 3.4
assert df_33['weight_kg'].isnull().sum() == 4 and df_33['battery_wh'].isnull().sum() == 59
assert df_33['weight_kg'].max() == 28.5 and df_33['range_km'].max() == 180
print('3.4 richtig ✔')

### 3.5 Aufbereitete Daten gegen die Quelle prüfen
`velos_clean.csv` wurde von jemand anderem aufbereitet und ist «inhaltlich nicht geprüft». Vergleiche die Preise:
Verbinde `df_33[['product_id', 'price_chf']]` mit `df_clean[['product_id', 'price_chf']]` über `product_id`
(Suffixe `_scraped` und `_clean`) und finde alle Produkte, deren Preise sich unterscheiden → `abweichungen`.

In [ ]:
# Import data from csv to pandas dataframe named 'df_clean'
df_clean = pd.read_csv(f'{DATA}/velos_clean.csv')

# Show dimensions and first rows
print('Number of rows and columns:', df_clean.shape)
df_clean.head()

In [ ]:
vergleich = df_33[['product_id', 'price_chf']].merge(df_clean[['product_id', 'price_chf']],
                                                     on='product_id', suffixes=('_scraped', '_clean'))
abweichungen = vergleich[(vergleich['price_chf_scraped'] - vergleich['price_chf_clean']).abs() > 0.5]
abweichungen
# VM-30032: im Shop CHF 3'098.–, in velos_clean.csv 30981 -> Tippfehler (eine Ziffer zu viel)

In [ ]:
# Selbstkontrolle 3.5
assert abweichungen['product_id'].tolist() == ['VM-30032']
print('3.5 richtig ✔')

---
## Stufe 4 — Fehlende Werte

### 4.1 Wo fehlt was?
Zähle die fehlenden Werte pro Spalte in `df_clean` → Series `fehlend` (nur Spalten mit mindestens einem fehlenden Wert).

In [ ]:
fehlend = df_clean.isnull().sum()
fehlend = fehlend[fehlend > 0]
fehlend

In [ ]:
# Selbstkontrolle 4.1
assert fehlend.to_dict() == {'weight_kg': 4, 'battery_wh': 59, 'range_km': 59, 'price_chf': 6}
print('4.1 richtig ✔')

### 4.2 Hängt das Fehlen von etwas ab?
Zähle pro `category`, wie viele Preise fehlen, und wie viele Akku-Angaben fehlen. → DataFrame `fehlend_kat` mit den
Spalten `preis_fehlt` und `akku_fehlt` (Index: `category`).

In [ ]:
fehlend_kat = df_clean.groupby('category').agg(
    preis_fehlt=('price_chf', lambda s: s.isnull().sum()),
    akku_fehlt=('battery_wh', lambda s: s.isnull().sum()))
fehlend_kat

In [ ]:
# Selbstkontrolle 4.2
assert int(fehlend_kat['preis_fehlt'].sum()) == 6
assert set(fehlend_kat[fehlend_kat['preis_fehlt'] > 0].index) == {'E-Mountainbike', 'S-Pedelec'}
assert int(fehlend_kat.loc['Rennvelo', 'akku_fehlt']) == 13
print('4.2 richtig ✔')

### 4.3 MCAR, MAR oder MNAR?
Zur Erinnerung:
- **MCAR** (missing completely at random): Das Fehlen hängt von nichts ab — reiner Zufall.
- **MAR** (missing at random): Das Fehlen hängt von **anderen, beobachteten** Merkmalen ab.
- **MNAR** (missing not at random): Das Fehlen hängt vom **fehlenden Wert selbst** ab.

Ordne zu und begründe mit den Ergebnissen aus 4.1 / 4.2:

a) `price_chf` fehlt, weil der Shop bei Velos über CHF 9'000 «Preis auf Anfrage» anzeigt.
b) `weight_kg` fehlt bei 4 Velos, weil der Lieferant zufällig einzelne Datenblätter nicht geliefert hat.
c) `battery_wh` fehlt bei 59 Velos — alle ohne Elektromotor.
d) Welchen Effekt hat `dropna()` bei a) auf den berichteten Durchschnittspreis?

**Ihre Antwort:**

a) **MNAR** — ob der Preis fehlt, hängt vom Preis selbst ab (nur sehr teure Velos). Die Häufung bei
E-Mountainbikes und S-Pedelecs (4.2) ist eine Folge davon.
b) **MCAR** — das Fehlen hängt weder vom Gewicht noch von anderen Merkmalen ab.
c) Eigentlich **kein** fehlender Wert im engeren Sinn: Ein Velo ohne Motor **hat** keinen Akku (strukturell fehlend,
«nicht anwendbar»). Formal hängt das Fehlen vollständig von der beobachteten Kategorie ab (MAR). Ein Auffüllen mit dem
Mittelwert wäre hier fachlich falsch.
d) Mit `dropna()` fallen gezielt die **teuersten** Velos weg. Der Durchschnittspreis wird dadurch **systematisch zu tief**
ausgewiesen (verzerrt, Bias), der Shop wirkt günstiger als er ist.

---
## Stufe 5 — EDA: Verteilung, Lagemasse, Ausreisser

### 5.1 Kennzahlen
Berechne für `df_clean['price_chf']` die Kennzahlen mit `describe()` und die Schiefe mit `skew()` → `kennzahlen`,
`schiefe` (auf 2 Stellen gerundet).

In [ ]:
kennzahlen = df_clean['price_chf'].describe().round(2)
schiefe = round(df_clean['price_chf'].skew(), 2)
print(kennzahlen, '\nSchiefe:', schiefe)

In [ ]:
# Selbstkontrolle 5.1
assert kennzahlen['count'] == 114 and kennzahlen['50%'] == 3748.0 and kennzahlen['max'] == 30981.0
assert abs(schiefe - 4.54) < 0.02
print('5.1 richtig ✔')

**Interpretation:**

Der Mittelwert (3808.59) liegt nur leicht über dem Median (3748), aber die Schiefe von ca. 4.5 und das Maximum von
30'981 (Q3 nur 4'873) zeigen einen **extremen Ausreisser** am rechten Rand. Er treibt Schiefe und Standardabweichung
stark nach oben. Der **Median** ist robust gegen Ausreisser und eignet sich besser als «typischer» Preis.

### 5.2 Verteilung visualisieren
Erstelle nebeneinander ein **Histogramm** (`sns.histplot`) und einen **Boxplot** (`sns.boxplot`) der Preise — mit
Titel und Achsenbeschriftungen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(data=df_clean, x='price_chf', bins=30, ax=axes[0])
axes[0].set_title('Histogramm der Velopreise')
axes[0].set_xlabel('Preis (CHF)')
axes[0].set_ylabel('Anzahl Velos')

sns.boxplot(data=df_clean, x='price_chf', ax=axes[1])
axes[1].set_title('Boxplot der Velopreise')
axes[1].set_xlabel('Preis (CHF)')

plt.tight_layout()
plt.show()

### 5.3 Ausreisser mit der IQR-Regel
Ein Wert gilt als Ausreisser, wenn er unter `Q1 − 1.5·IQR` oder über `Q3 + 1.5·IQR` liegt (`IQR = Q3 − Q1`).
Finde alle Ausreisser-Zeilen in `df_clean` (Spalten `product_id`, `brand`, `model`, `price_chf`) → `ausreisser`.

In [ ]:
q1 = df_clean['price_chf'].quantile(0.25)
q3 = df_clean['price_chf'].quantile(0.75)
iqr = q3 - q1
maske = (df_clean['price_chf'] < q1 - 1.5 * iqr) | (df_clean['price_chf'] > q3 + 1.5 * iqr)
ausreisser = df_clean.loc[maske, ['product_id', 'brand', 'model', 'price_chf']]
ausreisser

In [ ]:
# Selbstkontrolle 5.3
assert ausreisser['product_id'].tolist() == ['VM-30032']
print('5.3 richtig ✔')

### 5.4 Mittelwert oder Median pro Gruppe?
Berechne pro `brand` Mittelwert **und** Median des Preises (gerundet auf 2 Stellen), sortiert nach Mittelwert absteigend
→ `pro_marke`. Was fällt bei «Tour de Suisse» auf?

In [ ]:
pro_marke = (df_clean.groupby('brand')['price_chf']
             .agg(['mean', 'median']).round(2)
             .sort_values(by='mean', ascending=False))
pro_marke

In [ ]:
# Selbstkontrolle 5.4
assert pro_marke.loc['Tour de Suisse', 'mean'] == 4113.33 and pro_marke.loc['Tour de Suisse', 'median'] == 1598.0
print('5.4 richtig ✔')

**Beobachtung:**

Bei «Tour de Suisse» liegt der Mittelwert (4'113) mehr als doppelt so hoch wie der Median (1'598). Bei keiner anderen
Marke ist der Abstand so gross. Der einzelne Ausreisser `VM-30032` (30'981 statt 3'098) zieht den Mittelwert nach oben.
Die günstigste Marke erscheint so im Mittelfeld, vor Trek und Cube.

### 5.5 Ausreisser behandeln
Der Wert 30'981 ist nachweislich ein Tippfehler (Shop: 3'098, siehe 3.5). Korrigiere ihn in einer Kopie `df_55` und
berechne den Mittelwert pro Marke erneut (sortiert, gerundet) → `pro_marke_korr`.

*Alternative:* Ist der richtige Wert unbekannt, setzt man einen nachweislich falschen Wert auf `NaN` (und dokumentiert das).
Den Ausreisser **ungeprüft** zu löschen, ist dagegen keine gute Idee: Er könnte auch korrekt sein.

In [ ]:
df_55 = df_clean.copy()
df_55.loc[df_55['product_id'] == 'VM-30032', 'price_chf'] = 3098.0

pro_marke_korr = (df_55.groupby('brand')['price_chf'].mean().round(2)
                  .sort_values(ascending=False))
pro_marke_korr

In [ ]:
# Selbstkontrolle 5.5
assert pro_marke_korr.index[-1] == 'Tour de Suisse'
assert abs(pro_marke_korr['Tour de Suisse'] - 1789.75) < 0.01
print('5.5 richtig ✔')

---
## Stufe 6 — Klassen bilden: `cut`, `qcut`, `crosstab`

### 6.1 Feste Grenzen mit `pd.cut`
Teile die Velos nach `weight_kg` in die Klassen `leicht` (bis 10 kg), `mittel` (über 10 bis 20 kg) und `schwer`
(über 20 bis 30 kg) ein → neue Spalte `gewichtsklasse` in `df_6`. Zähle die Velos pro Klasse → `anzahl_klassen`.

In [ ]:
df_6 = df_clean.copy()
df_6['gewichtsklasse'] = pd.cut(df_6['weight_kg'], bins=[0, 10, 20, 30],
                                labels=['leicht', 'mittel', 'schwer'])

anzahl_klassen = df_6['gewichtsklasse'].value_counts().sort_index()
anzahl_klassen

In [ ]:
# Selbstkontrolle 6.1
assert anzahl_klassen.to_dict() == {'leicht': 35, 'mittel': 24, 'schwer': 57}
print('6.1 richtig ✔')

### 6.2 Gleich grosse Gruppen mit `pd.qcut`
Teile die Velos nach `price_chf` in **drei gleich grosse** Preisklassen `günstig`, `mittel`, `teuer` → Spalte
`segment`. Gib die Klassengrenzen aus (`retbins=True`). Was passiert mit Velos ohne Preis?

In [ ]:
df_6['segment'], grenzen = pd.qcut(df_6['price_chf'], q=3,
                                   labels=['günstig', 'mittel', 'teuer'], retbins=True)
print('Klassengrenzen:', grenzen)
print(df_6['segment'].value_counts(dropna=False))
# qcut bestimmt die Grenzen über die Quantile (hier 33.3 % und 66.7 %), damit jede Klasse etwa gleich viele Velos
# enthält. Velos ohne Preis erhalten keine Klasse (NaN).

In [ ]:
# Selbstkontrolle 6.2
assert df_6['segment'].notnull().sum() == 114 and df_6['segment'].isnull().sum() == 6
print('6.2 richtig ✔')

### 6.3 Kontingenztabelle
Erstelle mit `pd.crosstab` die Anzahl Velos pro `category` (Zeilen) und `segment` (Spalten) mit Zeilen- und
Spaltensummen (`margins=True`) → `kreuz`. Erstelle zusätzlich eine Variante mit Zeilenprozenten
(`normalize='index'`, gerundet auf 2 Stellen) → `kreuz_prozent`.

In [ ]:
kreuz = pd.crosstab(df_6['category'], df_6['segment'], margins=True)
kreuz_prozent = pd.crosstab(df_6['category'], df_6['segment'], normalize='index').round(2)
display(kreuz, kreuz_prozent)

In [ ]:
# Selbstkontrolle 6.3
assert kreuz.loc['All', 'All'] == 114
assert kreuz.loc['Kindervelo', 'günstig'] == 14
print('6.3 richtig ✔')

---
## Stufe 7 — JSON, `merge` und Korrelation

### 7.1 JSON einlesen
Lies `reviews.json` in den DataFrame `df_reviews` ein.

In [ ]:
df_reviews = pd.read_json(f'{DATA}/reviews.json')
df_reviews.head()

In [ ]:
# Selbstkontrolle 7.1
assert df_reviews.shape == (99, 3)
print('7.1 richtig ✔')

### 7.2 Welcher Join-Typ?
Verbinde `df_clean` mit `df_reviews` über `product_id` mit `how='inner'`, `'left'` und `'outer'`. Speichere jeweils die
Anzahl Zeilen in `n_inner`, `n_left`, `n_outer`. Untersuche mit `indicator=True`, wie viele Bewertungen zu Produkten
gehören, die **nicht mehr im Sortiment** sind → `n_nur_reviews`.

In [ ]:
n_inner = len(df_clean.merge(df_reviews, on='product_id', how='inner'))
n_left = len(df_clean.merge(df_reviews, on='product_id', how='left'))
outer = df_clean.merge(df_reviews, on='product_id', how='outer', indicator=True)
n_outer = len(outer)
n_nur_reviews = (outer['_merge'] == 'right_only').sum()
print(n_inner, n_left, n_outer, n_nur_reviews)
print(outer['_merge'].value_counts())

In [ ]:
# Selbstkontrolle 7.2
assert (n_inner, n_left, n_outer, n_nur_reviews) == (95, 120, 124, 4)
print('7.2 richtig ✔')

### 7.3 Pearson oder Spearman?
Berechne für die Velos **mit** Bewertung und **mit** Preis die Korrelation zwischen `price_chf` und `avg_score` —
einmal nach Pearson, einmal nach Spearman (je auf 2 Stellen gerundet) → `r_pearson`, `r_spearman`.

*Tipp:* `df[['price_chf', 'avg_score']].corr(method='spearman')` — fehlende Werte werden paarweise ignoriert.

In [ ]:
df_73 = df_clean.merge(df_reviews, on='product_id', how='inner')
r_pearson = round(df_73[['price_chf', 'avg_score']].corr(method='pearson').iloc[0, 1], 2)
r_spearman = round(df_73[['price_chf', 'avg_score']].corr(method='spearman').iloc[0, 1], 2)
print(r_pearson, r_spearman)

In [ ]:
# Selbstkontrolle 7.3
assert abs(r_pearson - 0.43) < 0.011 and abs(r_spearman - 0.34) < 0.011
print('7.3 richtig ✔')

**Erklärung:**

**Pearson** misst einen **linearen** Zusammenhang und reagiert stark auf Ausreisser (der Preis 30'981 hat grossen
Einfluss). **Spearman** rechnet mit **Rängen**: Sie misst, ob der Zusammenhang **monoton** ist, und ist robust gegen
Ausreisser und schiefe Verteilungen. Bei der rechtsschiefen Preisverteilung ist Spearman angemessener. Beide Werte
(ca. 0.3–0.4) zeigen einen schwachen bis mittleren positiven Zusammenhang: Teurere Velos werden tendenziell etwas
besser bewertet.

### 7.4 Wie verlässlich ist ein Durchschnitt aus 2 Bewertungen?
Einige Velos haben nur 1–3 Bewertungen. Berechne die Spearman-Korrelation nur für Velos mit **mindestens 5** Bewertungen
und Preis → `r_spearman_5` (gerundet), Anzahl Velos → `n_5`. Zeichne ein Streudiagramm (`sns.scatterplot`) mit Titel
und beschrifteten Achsen.

In [ ]:
df_74 = df_73[(df_73['review_count'] >= 5) & df_73['price_chf'].notnull()]
n_5 = len(df_74)
r_spearman_5 = round(df_74[['price_chf', 'avg_score']].corr(method='spearman').iloc[0, 1], 2)
print(n_5, r_spearman_5)

sns.scatterplot(data=df_74, x='price_chf', y='avg_score')
plt.title('Preis und Kundenbewertung (mind. 5 Bewertungen)')
plt.xlabel('Preis (CHF)')
plt.ylabel('Durchschnittliche Bewertung (1–5 Sterne)')
plt.show()

In [ ]:
# Selbstkontrolle 7.4
assert n_5 == 82 and abs(r_spearman_5 - 0.35) < 0.011
print('7.4 richtig ✔')

---
## Stufe 8 — Tabellen von einer Detailseite

Öffne `Data/velomarkt/html/product_detail.html` im Editor: Die Seite hat **drei** Tabellen mit derselben Klasse
`data-table`. Vor jeder Tabelle steht eine `<h2>`-Überschrift.

### 8.1 Die richtige Tabelle auswählen
1. Wie viele Tabellen gibt es? → `anzahl_tabellen`
2. Wähle die Tabelle **«Technische Daten»** aus, und zwar **über die Überschrift**, nicht über die Position: Suche das
   `h2`, dessen Text «Technische Daten» ist, und nimm die nächste Tabelle (`.find_next('table')`) → `table`

In [ ]:
with open(f'{DATA}/html/product_detail.html', 'r', encoding='utf-8') as file:
    soup_detail = BeautifulSoup(file.read(), 'html.parser')

anzahl_tabellen = len(soup_detail.find_all('table'))
ueberschrift = soup_detail.find('h2', string='Technische Daten')
table = ueberschrift.find_next('table')
# Alternative über die Position: soup_detail.find_all('table')[1] — funktioniert nur,
# solange sich die Reihenfolge der Tabellen auf der Seite nicht ändert.

In [ ]:
# Selbstkontrolle 8.1
assert anzahl_tabellen == 3 and 'Panasonic' in table.get_text()
print('8.1 richtig ✔')

### 8.2 Zwischenüberschriften entfernen
Der folgende Code überführt die Tabelle Zeile für Zeile in `df_82`. Schau dir das Ergebnis an: Die
Zwischenüberschriften (z. B. «Antrieb») haben nur **eine** Zelle, darum ist `wert` dort leer (`None`).
Entferne diese Zeilen und setze den Index neu.

In [ ]:
rows = []
for row in table.find_all('tr'):
    rows.append([cell.get_text().strip() for cell in row.find_all('td')])
df_82 = pd.DataFrame(rows, columns=['merkmal', 'wert'])
display(df_82)

df_82 = df_82.dropna(subset=['wert']).reset_index(drop=True)
# Alternative direkt beim Einlesen: Zeilen mit <td class="section"> überspringen
# oder nur Zeilen mit genau zwei <td> übernehmen.
df_82

In [ ]:
# Selbstkontrolle 8.2
assert len(df_82) == 9 and 'Antrieb' not in df_82['merkmal'].tolist()
print('8.2 richtig ✔')

### 8.3 Preisverlauf auswerten
Lies die Tabelle «Preisverlauf» in `df_83` (Spalten `monat`, `preis_raw`) und ergänze die Spalte `preis_chf` (Zahl).
Um wie viele CHF ist der Preis seit März gesunken? → `rueckgang`

In [ ]:
tabelle_preis = soup_detail.find('h2', string='Preisverlauf').find_next('table')
df_83 = pd.DataFrame([[td.get_text().strip() for td in tr.find_all('td')] for tr in tabelle_preis.find_all('tr')],
                     columns=['monat', 'preis_raw'])
df_83['preis_chf'] = df_83['preis_raw'].apply(preis_zu_zahl)
rueckgang = df_83['preis_chf'].iloc[0] - df_83['preis_chf'].iloc[-1]
df_83

In [ ]:
# Selbstkontrolle 8.3
assert rueckgang == 400
print('8.3 richtig ✔')

---
## Stufe 9 — Konzeptfragen (wie Prüfungsteil A)

Antworten zählen in der Prüfung nur **mit Bezug zu den Daten**. Nenne Spalten, Zahlen und konkrete Beispiele.

### 9.1 Korrelation ist nicht Kausalität
Ein Kollege findet zwischen `battery_wh` und `price_chf` (nur E-Bikes) einen Pearson-Koeffizienten von r = −0.10 und
folgert: *«Die Akkugrösse hat keinen Einfluss auf den Preis.»* Nenne zwei fachliche Einwände.

**Ihre Antwort:**

1. **Ausreisser / Datenfehler:** Pearson reagiert stark auf Extremwerte. Der Tippfehler `VM-30032` (30'981 statt
   3'098) ist ein E-Bike und verzerrt r. Erst bereinigen, oder robust mit Spearman rechnen.
2. **Fehlende Werte nicht zufällig (MNAR):** Die teuersten E-Bikes («Preis auf Anfrage», oft mit grossem Akku) fehlen
   in der Berechnung. Der Wertebereich ist abgeschnitten, das kann einen Zusammenhang verdecken.
3. **Störvariable / Vermischung:** E-City, E-Trekking, E-MTB und S-Pedelec haben ganz unterschiedliche Preisniveaus.
   Ein Zusammenhang innerhalb einer Kategorie kann durch das Mischen verschwinden. Zudem misst r nur **lineare**
   Zusammenhänge, und selbst ein hohes r würde keine Ursache belegen.

### 9.2 Verteilung beschreiben
Für `weight_kg` liefert pandas: `count 116, mean 17.30, std 7.55, min 6.80, 25% 9.70, 50% 16.40, 75% 24.42, max 28.50`
und `skew() = 0.00`. Eine Kollegin sagt: *«Schiefe 0 — die Gewichte sind schön symmetrisch um 17 kg verteilt.»*
Stimmt das? Welche Grafik hilft?

**Ihre Antwort:**

Schiefe 0 bedeutet nur **symmetrisch**, nicht «normalverteilt um den Mittelwert». Der grosse Abstand zwischen Q1 (9.7)
und Q3 (24.4) und die grosse Standardabweichung deuten auf **zwei Gruppen** hin: leichte Velos ohne Motor
(ca. 7–17 kg) und schwere E-Bikes (ca. 21–28 kg). Die Verteilung ist **bimodal**, und kaum ein Velo wiegt «typische»
17 kg. Ein **Histogramm** (oder ein Boxplot/Histogramm getrennt nach E-Bike ja/nein) macht das sichtbar. Statt
eines Gesamtmittelwerts berichtet man besser Kennzahlen pro Gruppe.

---
**Geschafft!** Weiter mit der Probeprüfung `21_probepruefung_ap02.ipynb` — mit Zeitlimit und ohne Lösung.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')